In [ ]:
# V7 : ablation H-J
import json
import pandas as pd
import gc
from pathlib import Path
import numpy as np

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "bigData"

# PATH_15m = Path.cwd().parents[1] / "data" / "bigData" / "BTCUSDT-15m.json"
PATH_5m =folder_path / "BTCUSDT-5m.json"

# with open(PATH_15m) as f:
#     file = json.load(f)

# ohlcv = file["ohlcv"]  # list of [timestamp, open, high, low, close, volume]
# del file  # free the rest of the JSON (symbol, onchart, etc.), , preserve RAM

# df_15 = pd.DataFrame(ohlcv, columns=["timestamp", "open", "high", "low", "close", "vol"])
# del ohlcv  # free the Python list-of-lists, preserve RAM
# gc.collect()

with open(PATH_5m) as f:
    file = json.load(f)

ohlcv = file["ohlcv"]  # list of [timestamp, open, high, low, close, volume]
del file  # free the rest of the JSON (symbol, onchart, etc.), , preserve RAM

df_5 = pd.DataFrame(ohlcv, columns=["timestamp", "open", "high", "low", "close", "vol"])
del ohlcv  # free the Python list-of-lists, preserve RAM
gc.collect()

# Halve memory: float64 -> float32 for price/volume columns
# df_15[["open", "high", "low", "close", "vol"]] = df_15[["open", "high", "low", "close", "vol"]].astype("float32")
df_5[["open", "high", "low", "close", "vol"]] = df_5[["open", "high", "low", "close", "vol"]].astype("float32")

# code was truncated by the NotebookEdit tool on the previous write
# print(f"15m Shape: {df_15.shape} | Memory: {df_15.memory_usage(deep=True).sum()/ 1e6:.2f} MB") # 2.15 MB
print(f"5m Shape: {df_5.shape} | Memory: {df_5.memory_usage(deep=True).sum()/ 1e6:.2f} MB") # 6.41 MB

# sort
df_5 = df_5.sort_values('timestamp').reset_index(drop=True)
# df_15 = df_15.sort_values('timestamp').reset_index(drop=True)

5m Shape: (228772, 6) | Memory: 6.41 MB


# 5m data - ATR42
## used to make triple barrier

In [6]:
# ATR42 on 5m — equivalent to ATR14 on 15m in time

prev_close_5m = df_5['close'].shift(1)
tr_5m = pd.concat([
    df_5['high'] - df_5['low'],
    (df_5['high'] - prev_close_5m).abs(),
    (df_5['low']  - prev_close_5m).abs(),
], axis=1).max(axis=1)

df_5['atr42'] = tr_5m.rolling(42).mean().astype('float32')

print(f"ATR42 range: {df_5['atr42'].min():.2f} – {df_5['atr42'].max():.2f}")
df_5[['timestamp', 'close', 'atr42']].tail(3)


ATR42 range: 7.42 – 1552.81


,timestamp,close,atr42
228769,1772668200000,72668.960938,162.623138
228770,1772668500000,72666.773438,161.973770
228771,1772668800000,72677.328125,162.421127


# Create label on 5m - Triple Barrier

In [7]:
# triple-barrier on 5m (gap-aware)
FIVE_MIN_MS = 300_000

# Snap 5m timestamps to grid (absorbs ms jitter) — needed for gap detection
df_5['ts_5m'] = (df_5['timestamp'] // FIVE_MIN_MS) * FIVE_MIN_MS

# Triple-barrier on 5m (gap-aware)
ATR_MULT    = 1.0
MAX_BARS_5M = 3   # 1h lookahead — tune: 6=30m, 12=1h, 24=2h, 48=4h

# df_5   = ltf_df.reset_index(drop=True)

high_arr  = df_5['high'].values
low_arr   = df_5['low'].values
open_arr  = df_5['open'].values
close_arr = df_5['close'].values
atr_arr   = df_5['atr42'].values
ts5_arr   = df_5['ts_5m'].values

labels = np.full(len(df_5), np.nan)

for i in range(len(df_5)):
    if np.isnan(atr_arr[i]):
        continue

    entry = close_arr[i]
    tp    = atr_arr[i] * ATR_MULT
    sl    = atr_arr[i] * ATR_MULT
    label = 0  # default: timeout

    for k in range(1, MAX_BARS_5M + 1):
        j = i + k
        if j >= len(df_5):
            break
        # Stop at data gap — don't evaluate across market closures
        if ts5_arr[j] != ts5_arr[i] + k * FIVE_MIN_MS:
            break

        h, l, o = high_arr[j], low_arr[j], open_arr[j]
        hit_up   = (h - entry) >= tp
        hit_down = (entry - l) >= sl

        if hit_up and hit_down:
            # Both barriers in same bar: open proximity infers direction first
            label = 1 if abs(o - (entry + tp)) < abs(o - (entry - sl)) else -1
            break
        elif hit_up:
            label = 1
            break
        elif hit_down:
            label = -1
            break

    labels[i] = label

df_5['label'] = labels

# Stats
total  = df_5['label'].notna().sum()
n_up   = (df_5['label'] ==  1).sum()
n_down = (df_5['label'] == -1).sum()
n_time = (df_5['label'] ==  0).sum()
n_nan  = df_5['label'].isna().sum()

print(f"Total labeled  : {total:,}")
print(f"  Up   ( 1)    : {n_up:,}  ({n_up   / total * 100:.1f}%)")
print(f"  Down (-1)    : {n_down:,}  ({n_down / total * 100:.1f}%)")
print(f"  Timeout (0)  : {n_time:,}  ({n_time  / total * 100:.1f}%)  ← masked during training")
print(f"  NaN (warmup) : {n_nan:,}")


Total labeled  : 228,731
  Up   ( 1)    : 76,220  (33.3%)
  Down (-1)    : 77,821  (34.0%)
  Timeout (0)  : 74,690  (32.7%)  ← masked during training
  NaN (warmup) : 41


In [8]:
# keep all rows, add train_mask
# Keep full sequence intact — LSTM needs contiguous rows
# train_mask = 1: valid label, backprop on this step
# train_mask = 0: timeout or warmup NaN, zero out loss contribution

df_5['train_mask'] = ((df_5['label'] == 1) | (df_5['label'] == -1)).astype('int8')

# Fill NaN warmup labels with 0 so the array stays numeric
df_5['label'] = df_5['label'].fillna(0).astype('int8')

trainable = df_5['train_mask'].sum()
total     = len(df_5)

print(f"Total rows     : {total:,}")
print(f"  Trainable    : {trainable:,}  ({trainable / total * 100:.1f}%)")
print(f"  Masked out   : {total - trainable:,}  (timeout + warmup NaN)")
masked_in = df_5[df_5['train_mask'] == 1]
print(f"Label balance (trainable): {masked_in['label'].value_counts().to_dict()}")
df_5.tail()



Total rows     : 228,772
  Trainable    : 154,041  (67.3%)
  Masked out   : 74,731  (timeout + warmup NaN)
Label balance (trainable): {-1: 77821, 1: 76220}


,timestamp,open,high,low,close,vol,atr42,ts_5m,label,train_mask
228767,1772667600000,72701.101562,72756.812500,72655.328125,72756.796875,56.436958,166.503159,1772667600000,0,0
228768,1772667900000,72756.812500,72830.007812,72701.179688,72701.187500,43.030441,164.771942,1772667900000,0,0
228769,1772668200000,72701.179688,72719.687500,72603.843750,72668.960938,169.471603,162.623138,1772668200000,1,1
228770,1772668500000,72668.953125,72775.468750,72639.679688,72666.773438,275.874146,161.973770,1772668500000,1,1
228771,1772668800000,72666.773438,72854.000000,72657.953125,72677.328125,168.545258,162.421127,1772668800000,0,0


# Export Data

In [ ]:
# Export marker to json visulize the data on Chart
# save export to jsonl
# Export marker to json visulize the data on Chart
# save export to jsonl

save_path = find_project_root() / "data" / "mlData"
save_path.mkdir(parents=True, exist_ok=True)

out_path = save_path / "BTCUSDT-5m-v7-H.jsonl"
df_5.to_json(out_path, orient="records", lines=True)
print(f"Saved {len(df_5):,} rows → {out_path}")

Saved 228,772 rows → /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-CandleScience/data/mlData/BTCUSD-5m-v7-H.jsonl
